# Notebook 00 — Setup & Path Verification

Run this notebook **once before anything else**.

The Results folder path is set in one place: `RESULTS_DIR` in `utils/data_utils.py`.  
It is currently set to `/ssd1/songjiangliu/shared/asset_clustering/Results`.  
To change it, edit that single line in `data_utils.py`.

**This notebook:**
1. Verifies the Results folder and both subfolders are reachable
2. Counts all available K × λ panel files
3. Smoke-tests the baseline panel (K=50, λ=1,000,000)
4. Shows which notebooks are ready to run

In [6]:
import os, sys
NOTEBOOK_DIR = os.path.abspath(os.getcwd())
REPO_ROOT    = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
sys.path.insert(0, REPO_ROOT)

# ── Results folder is read directly from data_utils.py ────────────
# To change the path: edit RESULTS_DIR in utils/data_utils.py
from utils.data_utils import RESULTS_DIR, K50_DIR, ALL_K_DIR

print(f"RESULTS_DIR : {RESULTS_DIR}")
print(f"  exists    : {os.path.isdir(RESULTS_DIR)}")
print()
print(f"K50_DIR     : {K50_DIR}")
print(f"  exists    : {os.path.isdir(K50_DIR)}")
print()
print(f"ALL_K_DIR   : {ALL_K_DIR}")
print(f"  exists    : {os.path.isdir(ALL_K_DIR)}")
print()
if not os.path.isdir(K50_DIR):
    print("!! K50_DIR not found.")
    print("   Edit RESULTS_DIR in utils/data_utils.py")
if not os.path.isdir(ALL_K_DIR):
    print("!! ALL_K_DIR not found.")
    print("   Edit RESULTS_DIR in utils/data_utils.py")

RESULTS_DIR : /ssd1/songjiangliu/shared/asset_clustering/Results
  exists    : True

K50_DIR     : /ssd1/songjiangliu/shared/asset_clustering/Results/cluster_month_panels_K_50
  exists    : True

ALL_K_DIR   : /ssd1/songjiangliu/shared/asset_clustering/Results/cluster_month_panels_all_K_except_50
  exists    : True



In [7]:
# ── Count available grid files ────────────────────────────────────
from utils.data_utils import K_GRID, RANKING_LAMBDA_STRS

found_k50  = []
found_allk = []
missing    = []

for K in K_GRID:
    for lam_str in RANKING_LAMBDA_STRS:
        for subdir in [K50_DIR, ALL_K_DIR]:
            p = os.path.join(subdir, f"cluster_month_panel_K_{K}_{lam_str}.csv")
            if os.path.exists(p):
                if K == 50: found_k50.append((K, lam_str))
                else:       found_allk.append((K, lam_str))
                break
        else:
            missing.append(f"K={K}, {lam_str}")

total   = len(K_GRID) * len(RANKING_LAMBDA_STRS)
n_found = len(found_k50) + len(found_allk)
print(f"Files found : {n_found} / {total}")
print(f"  K=50      : {len(found_k50)} / {len(RANKING_LAMBDA_STRS)}")
print(f"  Other K   : {len(found_allk)} / {(len(K_GRID)-1)*len(RANKING_LAMBDA_STRS)}")
if missing:
    print(f"  Missing ({len(missing)}): {missing[:3]}{'...' if len(missing)>3 else ''}")
else:
    print("  No missing files — full grid available!")

Files found : 301 / 351
  K=50      : 13 / 13
  Other K   : 288 / 338
  Missing (50): ['K=2, lambda_1e-08', 'K=2, lambda_1e-09', 'K=3, lambda_1e-08']...


In [8]:
# ── Smoke-test baseline panel ─────────────────────────────────────
from utils.data_utils import (load_cluster_panel, load_cluster_ranking,
                               pivot_and_rank, ann_sharpe)
import pandas as pd

try:
    ranking = load_cluster_ranking()
    df      = load_cluster_panel(K=50, lam=1_000_000)
    cr, _   = pivot_and_rank(df, lam=1_000_000, ranking_df=ranking)
    print("Baseline panel (K=50, λ=1e6) loaded successfully")
    print(f"  Shape        : {cr.shape}  ({cr.shape[0]} months × {cr.shape[1]} clusters)")
    print(f"  Period       : {cr.index[0]}  →  {cr.index[-1]}")
    print(f"  L01 mean     : {cr['L01'].mean():.4f}  (paper: -0.0185)")
    print(f"  L50 mean     : {cr['L50'].mean():.4f}  (paper: +0.0158)")
    print(f"  H-L spread   : {cr['L50'].mean()-cr['L01'].mean():.4f}  (paper: +0.0343)")
except Exception as e:
    print(f"Error: {e}")

Baseline panel (K=50, λ=1e6) loaded successfully
  Shape        : (511, 50)  (511 months × 50 clusters)
  Period       : 1977-01-01 00:00:00  →  2019-12-01 00:00:00
  L01 mean     : -0.0192  (paper: -0.0185)
  L50 mean     : 0.0159  (paper: +0.0158)
  H-L spread   : 0.0350  (paper: +0.0343)


In [9]:
# ── Notebook readiness summary ────────────────────────────────────
import os
DATA_DIR = os.path.join(REPO_ROOT, "data")

aux = {
    "factor_returns.csv":   os.path.join(DATA_DIR, "factor_returns.csv"),
    "macro_predictors.csv": os.path.join(DATA_DIR, "macro_predictors.csv"),
    "mebm_25.csv":          os.path.join(DATA_DIR, "mebm_25.csv"),
    "benchmark_sorts.csv":  os.path.join(DATA_DIR, "benchmark_sorts.csv"),
    "centroid_chars.csv":   os.path.join(DATA_DIR, "centroid_chars.csv"),
    "firm_panel.csv":       os.path.join(DATA_DIR, "firm_panel.csv"),
}
aux_ok = {k: os.path.exists(v) for k, v in aux.items()}

has_baseline = len(found_k50) > 0
has_full_k50 = len(found_k50) == len(RANKING_LAMBDA_STRS)
has_allk     = len(found_allk) > 0

notebooks = [
    ("01_Table1",         "Table 1",              has_baseline),
    ("02_Figure3",        "Figure 3 (full grid)", has_full_k50 and has_allk),
    ("03_Figures4_5_6",   "Figures 4, 5, 6",      has_baseline),
    ("04_Tables2_3",      "Tables 2 & 3",         has_baseline and aux_ok["factor_returns.csv"]),
    ("05_Tables5_6_9_10", "Tables 5,6 / Figs 9,10", aux_ok["centroid_chars.csv"]),
    ("06_Figs11_12_13",   "Figures 11-13",        aux_ok["centroid_chars.csv"]),
    ("07_Table4",         "Table 4",              aux_ok["firm_panel.csv"]),
    ("08_Figs7_8",        "Figures 7 & 8",        aux_ok["firm_panel.csv"]),
    ("09_Fig14_Table7",   "Figure 14, Table 7",   aux_ok["firm_panel.csv"]),
    ("10_Figs15_16",      "Figures 15 & 16",      has_baseline and aux_ok["factor_returns.csv"] and aux_ok["mebm_25.csv"]),
    ("11_Table8",         "Table 8",              has_baseline and aux_ok["factor_returns.csv"] and aux_ok["macro_predictors.csv"]),
]
print("=" * 55)
print("NOTEBOOK READINESS")
print("=" * 55)
for nb_id, desc, ready in notebooks:
    icon = "READY  " if ready else "needs more data"
    print(f"  {'v' if ready else 'x'} {icon}  {desc}")
print()
print("Auxiliary CSVs go in:  replication_repo/data/")
for fname, ok in aux_ok.items():
    print(f"  {'v' if ok else 'x'} {fname}")

NOTEBOOK READINESS
  v READY    Table 1
  v READY    Figure 3 (full grid)
  v READY    Figures 4, 5, 6
  v READY    Tables 2 & 3
  x needs more data  Tables 5,6 / Figs 9,10
  x needs more data  Figures 11-13
  x needs more data  Table 4
  x needs more data  Figures 7 & 8
  x needs more data  Figure 14, Table 7
  x needs more data  Figures 15 & 16
  x needs more data  Table 8

Auxiliary CSVs go in:  replication_repo/data/
  v factor_returns.csv
  x macro_predictors.csv
  x mebm_25.csv
  x benchmark_sorts.csv
  x centroid_chars.csv
  x firm_panel.csv
